# 93 Execution Readiness Checklist Reference

Reference-only notebook. It is an optional structural repo check and is not required in the main notebook run order.


In [ ]:
from pathlib import Path
import importlib.util
import os
import subprocess
import sys
import time

import pandas as pd
from IPython.display import Markdown, display

NOTEBOOK_CWD = Path.cwd()
REPO_ROOT = next(
    path for path in [NOTEBOOK_CWD, *NOTEBOOK_CWD.parents] if (path / "scripts/Data/02_Forecasting/01_DA_prices").exists()
)
os.chdir(REPO_ROOT)
PACKAGE_ROOT = REPO_ROOT / "scripts/Data/02_Forecasting/01_DA_prices"
if str(PACKAGE_ROOT) not in sys.path:
    sys.path.append(str(PACKAGE_ROOT))

from hourly_da.core.config import HourlyDAPipelineConfig
from hourly_da.core.external_features import build_external_family_catalog, load_external_feature_store
from hourly_da.core.methodology import (
    STARTER_ENDOGENOUS_FEATURE_NOTE,
    feature_stage_policy_frame,
    model_status_frame,
    shortlisting_policy_frame,
)
from hourly_da.core.reporting import find_latest_run, load_csv, load_json
from hourly_da.core.tuning import build_tuning_placeholder, tuning_cadence_frame, tuning_snippet_frame
from hourly_da.models import naive_model_names
from hourly_da.notebook_support import estimate_run_duration_seconds, format_duration, load_selected_case_weeks

config = HourlyDAPipelineConfig(
    input_csv=REPO_ROOT / "data/01_cleaned/Day_ahead_prices/DA_prices/hourly/da_prices_all_regions_hourly.csv",
    raw_root=REPO_ROOT / "data/00_Raw/DA_Prices",
    cleaned_feature_root=REPO_ROOT / "data/01_cleaned",
    output_root=REPO_ROOT / "data/02_Forecasting/01_DA_prices/hourly_da",
)
output_root = config.output_root


def latest_run_or_none(run_label: str) -> Path | None:
    try:
        return find_latest_run(output_root, run_label)
    except FileNotFoundError:
        return None


def ensure_required_modules(module_names: list[str], install_command: str | None = None) -> None:
    missing = [module_name for module_name in module_names if importlib.util.find_spec(module_name) is None]
    if not missing:
        return

    message_lines = [
        "Missing required package(s) in the active notebook interpreter: " + ", ".join(missing),
        f"Active interpreter: {sys.executable}",
    ]
    if install_command:
        message_lines.append(f"Install command: {install_command}")
    raise RuntimeError("\n".join(message_lines))


def run_command_with_live_output(command: list[str]) -> None:
    print("Running command:")
    print(" ".join(str(part) for part in command))
    process = subprocess.Popen(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        encoding="utf-8",
        errors="replace",
        bufsize=1,
    )

    output_tail: list[str] = []
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="")
        output_tail.append(line)
        if len(output_tail) > 40:
            output_tail.pop(0)

    return_code = process.wait()
    if return_code != 0:
        tail_text = "".join(output_tail).strip()
        message = f"Command failed with exit code {return_code}."
        if tail_text:
            message += "\nLast output:\n" + tail_text
        raise RuntimeError(message)


In [ ]:
paths_to_check = [
    PACKAGE_ROOT / "run_naive_benchmark.py",
    PACKAGE_ROOT / "run_lear_benchmark.py",
    PACKAGE_ROOT / "run_xgboost_benchmark.py",
    PACKAGE_ROOT / "run_prophet_benchmark.py",
    PACKAGE_ROOT / "run_model_comparison.py",
    PACKAGE_ROOT / "docs" / "feature_sets.md",
    PACKAGE_ROOT / "docs" / "tuning_policy.md",
    PACKAGE_ROOT / "hourly_da" / "core" / "methodology.py",
    PACKAGE_ROOT / "hourly_da" / "core" / "tuning.py",
    PACKAGE_ROOT / "hourly_da" / "models" / "registry.py",
    PACKAGE_ROOT / "hourly_da" / "models" / "prophet_model.py",
]

check_rows = []
for path in paths_to_check:
    check_rows.append({"path": str(path.relative_to(REPO_ROOT)), "exists": path.exists()})

display(pd.DataFrame(check_rows))


In [ ]:
rows = []
for run_label in ['case_week_selection', 'naive_benchmark', 'lear_fs1_benchmark', 'xgboost_fs1_benchmark', 'fs1_model_comparison', 'lear_fs2_benchmark', 'xgboost_fs2_benchmark', 'prophet_benchmark', 'model_comparison']:
    run_dir = latest_run_or_none(run_label)
    rows.append(
        {
            "run_label": run_label,
            "latest_run": str(run_dir) if run_dir is not None else "not yet run",
        }
    )

display(pd.DataFrame(rows))


If the files above exist, the methodology tables are centralized, and the run labels are recognized, the repo is structurally ready.
